# CropCare AI — Colab GPU 학습 / 파인튜닝

사용법:
1. 메뉴 **런타임 > 런타임 유형 변경 > T4 GPU** 선택
2. 위에서부터 순서대로 실행. 두 번째 셀에서 로컬 `ml/colab_bundle.zip` 업로드
   (`pack_for_colab.py`가 dataset + train.py + 현재 best_model.pth를 함께 묶음)
3. **처음부터 학습**은 3-A 셀, **정확도 개선 파인튜닝**은 3-B 셀 중 하나만 실행
4. 마지막 셀이 결과 체크포인트를 다운로드 → 로컬 `ml/checkpoints/`에 반영

3-B(파인튜닝)는 소수 클래스 균형 샘플링 + 256px 해상도로 무 질병 혼동을 집중 개선하고,
기존 best_model.pth에서 이어학습해 val 정확도가 더 좋을 때만 저장합니다.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU만 사용 가능 — 런타임 유형을 GPU로 바꾸세요')

In [ ]:
# colab_bundle.zip 업로드 (dataset + train.py + checkpoints/best_model.pth)
from google.colab import files
uploaded = files.upload()
!unzip -q -o colab_bundle.zip
!ls dataset/train && ls checkpoints

In [ ]:
# 3-A) 처음부터 학습 (베이스라인 재현용)
!python train.py --data-dir dataset --arch resnet50 --epochs 20 --batch-size 64 --lr 1e-4 --num-workers 2 --output checkpoints/best_model.pth

In [ ]:
# 3-B) 정확도 개선 파인튜닝: 소수 클래스 균형 + 256px, 기존 체크포인트 이어학습
# 결과는 best_model_ft.pth 로 저장(원본 보존). val이 좋을 때만 갱신됨.
!python train.py --data-dir dataset --arch resnet50 \n    --resume checkpoints/best_model.pth --balanced-sampler --img-size 256 \n    --lr 2e-5 --epochs 12 --batch-size 48 --num-workers 2 \n    --output checkpoints/best_model_ft.pth

In [ ]:
# 결과 체크포인트 다운로드 (3-A면 best_model.pth, 3-B면 best_model_ft.pth)
from google.colab import files
import os
path = 'checkpoints/best_model_ft.pth' if os.path.exists('checkpoints/best_model_ft.pth') else 'checkpoints/best_model.pth'
print('downloading', path)
files.download(path)